# Introduction

<center><img src="https://i.imgur.com/9hLRsjZ.jpg" height=400></center>

This dataset was scraped from [nextspaceflight.com](https://nextspaceflight.com/launches/past/?page=1) and includes all the space missions since the beginning of Space Race between the USA and the Soviet Union in 1957!

### Install Package with Country Codes

### Upgrade Plotly

Run the cell below if you are working with Google Colab.

In [34]:
%pip install --upgrade plotly

Note: you may need to restart the kernel to use updated packages.


c:\python_envs\central_env\Scripts\python.exe: No module named pip


### Import Statements

In [35]:
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

# These might be helpful:
from iso3166 import countries
from datetime import datetime, timedelta

### Notebook Presentation

In [36]:
pd.options.display.float_format = '{:,.2f}'.format

### Load the Data

In [37]:
df_data = pd.read_csv('mission_launches.csv')

# Preliminary Data Exploration

* What is the shape of `df_data`? 
* How many rows and columns does it have?
* What are the column names?
* Are there any NaN values or duplicates?

In [38]:
df_data.shape

(4324, 9)

In [39]:
df_data.columns, df_data.isna().values.any()

(Index(['Unnamed: 0.1', 'Unnamed: 0', 'Organisation', 'Location', 'Date',
        'Detail', 'Rocket_Status', 'Price', 'Mission_Status'],
       dtype='str'),
 np.True_)

## Data Cleaning - Check for Missing Values and Duplicates

Consider removing columns containing junk data. 

In [70]:
df_data_clean = df_data.drop_duplicates().copy()
# Price is the only column with missing values, and it's legitimately
# unreported for many missions — don't drop rows for it here.

In [71]:
df_data_clean = df_data.dropna()

## Descriptive Statistics

In [72]:
df_data_clean

,Unnamed: 0.1,Unnamed: 0,Organisation,Location,Date,Detail,Rocket_Status,Price,Mission_Status
0,0,0,SpaceX,"LC-39A, Kennedy Space Center, Florida, USA","Fri Aug 07, 2020 05:12 UTC",Falcon 9 Block 5 | Starlink V1 L9 & BlackSky,StatusActive,50.0,Success
1,1,1,CASC,"Site 9401 (SLS-2), Jiuquan Satellite Launch Ce...","Thu Aug 06, 2020 04:01 UTC",Long March 2D | Gaofen-9 04 & Q-SAT,StatusActive,29.75,Success
3,3,3,Roscosmos,"Site 200/39, Baikonur Cosmodrome, Kazakhstan","Thu Jul 30, 2020 21:25 UTC",Proton-M/Briz-M | Ekspress-80 & Ekspress-103,StatusActive,65.0,Success
4,4,4,ULA,"SLC-41, Cape Canaveral AFS, Florida, USA","Thu Jul 30, 2020 11:50 UTC",Atlas V 541 | Perseverance,StatusActive,145.0,Success
5,5,5,CASC,"LC-9, Taiyuan Satellite Launch Center, China","Sat Jul 25, 2020 03:13 UTC","Long March 4B | Ziyuan-3 03, Apocalypse-10 & N...",StatusActive,64.68,Success
...,...,...,...,...,...,...,...,...,...
3855,3855,3855,US Air Force,"SLC-4W, Vandenberg AFB, California, USA","Fri Jul 29, 1966 18:43 UTC",Titan IIIB | KH-8,StatusRetired,59.0,Success
3971,3971,3971,US Air Force,"SLC-20, Cape Canaveral AFS, Florida, USA","Thu May 06, 1965 15:00 UTC",Titan IIIA | LES 2 & LCS 1,StatusRetired,63.23,Success
3993,3993,3993,US Air Force,"SLC-20, Cape Canaveral AFS, Florida, USA","Thu Feb 11, 1965 15:19 UTC",Titan IIIA | LES 1,StatusRetired,63.23,Success
4000,4000,4000,US Air Force,"SLC-20, Cape Canaveral AFS, Florida, USA","Thu Dec 10, 1964 16:52 UTC",Titan IIIA | Transtage 2,StatusRetired,63.23,Success


In [43]:
df_data_clean.shape

(964, 9)

# Number of Launches per Company

Create a chart that shows the number of space mission launches by organisation.

In [44]:
launches_by_organisation = (
    df_data_clean['Organisation']
    .value_counts()
    .rename_axis('Organisation')
    .reset_index(name='Launches')
)

launches_by_organisation.head()

,Organisation,Launches
0,CASC,158
1,NASA,149
2,SpaceX,99
3,ULA,98
4,Arianespace,96


In [45]:
fig = px.bar(
    launches_by_organisation,
    x='Launches',
    y='Organisation',
    orientation='h',
    title='Number of Space Mission Launches by Organisation',
    labels={'Launches': 'Number of launches', 'Organisation': 'Organisation'},
    color='Launches',
    color_continuous_scale='matter'
)

fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    coloraxis_showscale=False,
    height=800
)

from IPython.display import HTML
HTML(fig.to_html(include_plotlyjs='cdn'))

# Number of Active versus Retired Rockets

How many rockets are active compared to those that are decomissioned? 

In [46]:
df_data_clean[df_data_clean["Rocket_Status"] == "StatusActive"].shape

(586, 9)

In [47]:
df_data_clean[df_data_clean["Rocket_Status"] == "StatusRetired"].shape

(378, 9)

# Distribution of Mission Status

How many missions were successful?
How many missions failed?

In [48]:
df_data_clean[df_data_clean["Mission_Status"] == "Success"].shape

(910, 9)

In [49]:
df_data_clean[df_data_clean["Mission_Status"] == "Failure"].shape

(36, 9)

# How Expensive are the Launches? 

Create a histogram and visualise the distribution. The price column is given in USD millions (careful of missing values). 

In [50]:
priced_launches = df_data.drop_duplicates().copy()
priced_launches['Price'] = pd.to_numeric(priced_launches['Price'], errors='coerce')
priced_launches = priced_launches.dropna(subset=['Price'])

priced_launches['Price'].describe()

count   949.00
mean    129.80
std     143.22
min       5.30
25%      40.00
50%      62.00
75%     164.00
max     450.00
Name: Price, dtype: float64

In [51]:
fig = px.histogram(
    priced_launches,
    x='Price',
    nbins=30,
    title='Distribution of Space Mission Launch Prices',
    labels={'Price': 'Launch price (USD millions)', 'count': 'Number of launches'},
    color_discrete_sequence=['#7b2d26']
)

fig.update_layout(
    xaxis_title='Launch price (USD millions)',
    yaxis_title='Number of launches',
    bargap=0.05,
    height=550
)

HTML(fig.to_html(include_plotlyjs='cdn'))

# Use a Choropleth Map to Show the Number of Launches by Country

* Create a choropleth map using [the plotly documentation](https://plotly.com/python/choropleth-maps/)
* Experiment with [plotly's available colours](https://plotly.com/python/builtin-colorscales/). I quite like the sequential colour `matter` on this map. 
* You'll need to extract a `country` feature as well as change the country names that no longer exist.

Wrangle the Country Names

You'll need to use a 3 letter country code for each country. You might have to change some country names.

* Russia is the Russian Federation
* New Mexico should be USA
* Yellow Sea refers to China
* Shahrud Missile Test Site should be Iran
* Pacific Missile Range Facility should be USA
* Barents Sea should be Russian Federation
* Gran Canaria should be USA


You can use the iso3166 package to convert the country names to Alpha3 format.

In [52]:
df_map = df_data_clean.copy()

country_replacements = {
    'Russia': 'Russian Federation',
    'New Mexico': 'USA',
    'Yellow Sea': 'China',
    'Shahrud Missile Test Site': 'Iran',
    'Pacific Missile Range Facility': 'USA',
    'Barents Sea': 'Russian Federation',
    'Gran Canaria': 'USA',
    'South Korea': 'Korea, Republic of',
    'North Korea': 'Korea, Democratic People\'s Republic of',
    'USA': 'United States',
    'UK': 'United Kingdom'
}

df_map['Country'] = (
    df_map['Location']
    .str.split(',')
    .str[-1]
    .str.strip()
    .replace(country_replacements)
)

country_lookup = {country.name: country.alpha3 for country in countries}
country_code_overrides = {
    'United States': 'USA',
    'United Kingdom': 'GBR',
    'Korea, Republic of': 'KOR',
    "Korea, Democratic People's Republic of": 'PRK'
}
df_map['Country Code'] = (
    df_map['Country']
    .map(country_lookup)
    .fillna(df_map['Country'].map(country_code_overrides))
)
df_map[['Location', 'Country', 'Country Code']].head()

,Location,Country,Country Code
0,"LC-39A, Kennedy Space Center, Florida, USA",United States,USA
1,"Site 9401 (SLS-2), Jiuquan Satellite Launch Ce...",China,CHN
3,"Site 200/39, Baikonur Cosmodrome, Kazakhstan",Kazakhstan,KAZ
4,"SLC-41, Cape Canaveral AFS, Florida, USA",United States,USA
5,"LC-9, Taiyuan Satellite Launch Center, China",China,CHN


In [53]:
launches_by_country = (
    df_map
    .dropna(subset=['Country Code'])
    .groupby(['Country', 'Country Code'])
    .size()
    .reset_index(name='Launches')
    .sort_values('Launches', ascending=False)
)

fig = px.choropleth(
    launches_by_country,
    locations='Country Code',
    color='Launches',
    hover_name='Country',
    color_continuous_scale='matter',
    title='Number of Space Mission Launches by Country'
)

fig.update_layout(
    geo=dict(showframe=False, showcoastlines=True),
    height=650
)

HTML(fig.to_html(include_plotlyjs='cdn'))

# Use a Choropleth Map to Show the Number of Failures by Country


In [54]:
failures_by_country = (
    df_map[df_map['Mission_Status'].str.contains('Failure', na=False)]
    .dropna(subset=['Country Code'])
    .groupby(['Country', 'Country Code'])
    .size()
    .reset_index(name='Failures')
    .sort_values('Failures', ascending=False)
)

fig = px.choropleth(
    failures_by_country,
    locations='Country Code',
    color='Failures',
    hover_name='Country',
    color_continuous_scale='matter',
    title='Number of Space Mission Failures by Country'
)

fig.update_layout(
    geo=dict(showframe=False, showcoastlines=True),
    height=650
)

HTML(fig.to_html(include_plotlyjs='cdn'))

# Create a Plotly Sunburst Chart of the countries, organisations, and mission status. 

In [55]:
sunburst_data = (
    df_map[['Country', 'Organisation', 'Mission_Status']]
    .dropna()
    .copy()
)

sunburst_data.head()

,Country,Organisation,Mission_Status
0,United States,SpaceX,Success
1,China,CASC,Success
3,Kazakhstan,Roscosmos,Success
4,United States,ULA,Success
5,China,CASC,Success


In [56]:
fig = px.sunburst(
    sunburst_data,
    path=['Country', 'Organisation', 'Mission_Status'],
    title='Space Missions by Country, Organisation, and Mission Status',
    color='Mission_Status',
    color_discrete_map={
        'Success': '#2a9d8f',
        'Failure': '#e76f51',
        'Partial Failure': '#f4a261',
        'Prelaunch Failure': '#9b2226'
    }
)

fig.update_layout(
    height=750,
    margin=dict(t=70, l=10, r=10, b=10)
)

fig_json = fig.to_json()

In [57]:
HTML(fig.to_html(include_plotlyjs='cdn'))

# Analyse the Total Amount of Money Spent by Organisation on Space Missions

In [58]:
priced_launches = df_data.drop_duplicates().copy()
priced_launches['Price'] = pd.to_numeric(priced_launches['Price'], errors='coerce')
priced_launches = priced_launches.dropna(subset=['Price'])

total_spending = priced_launches['Price'].sum()
total_spending

np.float64(123175.68)

In [59]:
spending_by_organisation = (
    priced_launches
    .groupby('Organisation', as_index=False)['Price']
    .sum()
    .rename(columns={'Price': 'Total Spending'})
    .sort_values('Total Spending', ascending=False)
)

spending_by_organisation.head(10)

,Organisation,Total Spending
14,NASA,"61,200.00"
0,Arianespace,"16,345.00"
20,ULA,"14,798.00"
2,CASC,"6,340.26"
19,SpaceX,"5,444.00"
15,Northrop,"3,930.00"
12,MHI,"3,532.50"
8,ISRO,"2,177.00"
21,US Air Force,"1,550.92"
22,VKS RF,"1,548.90"


In [60]:
fig = px.bar(
    spending_by_organisation,
    x='Total Spending',
    y='Organisation',
    orientation='h',
    title='Total Money Spent on Space Missions by Organisation',
    labels={
        'Total Spending': 'Total spending (USD millions)',
        'Organisation': 'Organisation'
    },
    color='Total Spending',
    color_continuous_scale='matter'
)

fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    coloraxis_showscale=False,
    height=800
)

HTML(fig.to_html(include_plotlyjs='cdn'))

# Analyse the Amount of Money Spent by Organisation per Launch

In [61]:
spending_per_launch = (
    priced_launches
    .groupby('Organisation', as_index=False)
    .agg(
        Total_Spending=('Price', 'sum'),
        Priced_Launches=('Price', 'count')
    )
)

spending_per_launch['Average_Spending_Per_Launch'] = (
    spending_per_launch['Total_Spending']
    / spending_per_launch['Priced_Launches']
)

spending_per_launch = spending_per_launch.sort_values(
    'Average_Spending_Per_Launch',
    ascending=False
)

spending_per_launch.head(10)

,Organisation,Total_Spending,Priced_Launches,Average_Spending_Per_Launch
14,NASA,"61,200.00",136,450.00
1,Boeing,"1,241.00",7,177.29
0,Arianespace,"16,345.00",96,170.26
20,ULA,"14,798.00",98,151.00
7,ILS,"1,320.00",13,101.54
12,MHI,"3,532.50",37,95.47
13,Martin Marietta,721.40,9,80.16
21,US Air Force,"1,550.92",26,59.65
9,JAXA,168.00,3,56.00
19,SpaceX,"5,444.00",99,54.99


In [62]:
fig = px.bar(
    spending_per_launch,
    x='Average_Spending_Per_Launch',
    y='Organisation',
    orientation='h',
    title='Average Money Spent per Launch by Organisation',
    labels={
        'Average_Spending_Per_Launch': 'Average spending per launch (USD millions)',
        'Organisation': 'Organisation'
    },
    color='Average_Spending_Per_Launch',
    color_continuous_scale='matter',
    hover_data=['Total_Spending', 'Priced_Launches']
)

fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    coloraxis_showscale=False,
    height=800
)

fig_json = fig.to_json()

In [63]:
HTML(fig.to_html(include_plotlyjs='cdn'))

# Chart the Number of Launches per Year

In [64]:
launches_by_year = df_data.drop_duplicates().copy()
launches_by_year['Date'] = pd.to_datetime(
    launches_by_year['Date'],
    errors='coerce'
)
launches_by_year = (
    launches_by_year
    .dropna(subset=['Date'])
    .assign(Year=lambda data: data['Date'].dt.year)
    .groupby('Year')
    .size()
    .reset_index(name='Launches')
    .sort_values('Year')
)

launches_by_year.head()

,Year,Launches
0,1957,3
1,1958,22
2,1959,20
3,1960,38
4,1961,52


In [65]:
fig = px.bar(
    launches_by_year,
    x='Year',
    y='Launches',
    title='Number of Space Mission Launches per Year',
    labels={
        'Year': 'Year',
        'Launches': 'Number of launches'
    },
    color='Launches',
    color_continuous_scale='matter'
)

fig.update_layout(
    coloraxis_showscale=False,
    height=550,
    bargap=0.1
)

HTML(fig.to_html(include_plotlyjs='cdn'))

# Chart the Number of Launches Month-on-Month until the Present

Which month has seen the highest number of launches in all time? Superimpose a rolling average on the month on month time series chart. 

In [ ]:
roll_df = df_data.drop_duplicates().copy()
roll_df['Date'] = pd.to_datetime(
    roll_df['Date'],
    errors='coerce'
)
roll_df = (
    roll_df
    .dropna(subset=['Date'])
    .assign(Year=lambda data: data['Date'].dt.year)
    .groupby('Year')
    .size()
    .reset_index(name='Launches')
    .sort_values('Year')
)

roll_df['Rolling_Avg'] = (
    roll_df['Launches']
    .rolling(window=5, min_periods=1)
    .mean()
)

In [69]:
fig = px.line(
    roll_df,
    x='Year',
    y='Rolling_Avg',
    title='5-Year Rolling Average of Space Mission Launches per Year',
    labels={
        'Year': 'Year',
        'Rolling_Avg': '5-Year Rolling Average of Launches'
    }
)

fig.update_layout(
    height=550
)

HTML(fig.to_html(include_plotlyjs='cdn'))

# Launches per Month: Which months are most popular and least popular for launches?

Some months have better weather than others. Which time of year seems to be best for space missions?

In [73]:
launches_by_month = df_data.drop_duplicates().copy()
launches_by_month['Date'] = pd.to_datetime(
    launches_by_month['Date'],
    errors='coerce',
    utc=True
)
launches_by_month = launches_by_month.dropna(subset=['Date'])
launches_by_month['Month'] = launches_by_month['Date'].dt.month

launches_by_month = (
    launches_by_month
    .groupby('Month')
    .size()
    .reset_index(name='Launches')
    .sort_values('Month')
)

month_names = {
    1: 'January', 2: 'February', 3: 'March', 4: 'April',
    5: 'May', 6: 'June', 7: 'July', 8: 'August',
    9: 'September', 10: 'October', 11: 'November', 12: 'December'
}
launches_by_month['Month_Name'] = launches_by_month['Month'].map(month_names)

launches_by_month

,Month,Launches,Month_Name
0,1,265,January
1,2,323,February
2,3,349,March
3,4,366,April
4,5,320,May
5,6,386,June
6,7,347,July
7,8,358,August
8,9,354,September
9,10,375,October


In [74]:
fig = px.bar(
    launches_by_month,
    x='Month_Name',
    y='Launches',
    title='Number of Space Mission Launches by Month',
    labels={'Month_Name': 'Month', 'Launches': 'Number of launches'},
    color='Launches',
    color_continuous_scale='matter',
    category_orders={'Month_Name': list(month_names.values())}
)

fig.update_layout(
    coloraxis_showscale=False,
    height=550,
    bargap=0.1
)

HTML(fig.to_html(include_plotlyjs='cdn'))

# How has the Launch Price varied Over Time? 

Create a line chart that shows the average price of rocket launches over time. 

In [75]:
price_over_time = df_data.drop_duplicates().copy()
price_over_time['Date'] = pd.to_datetime(
    price_over_time['Date'],
    errors='coerce',
    utc=True
)
price_over_time['Price'] = pd.to_numeric(price_over_time['Price'], errors='coerce')
price_over_time = price_over_time.dropna(subset=['Date', 'Price'])
price_over_time['Year'] = price_over_time['Date'].dt.year

avg_price_by_year = (
    price_over_time
    .groupby('Year', as_index=False)['Price']
    .mean()
    .rename(columns={'Price': 'Average_Price'})
    .sort_values('Year')
)

avg_price_by_year

,Year,Average_Price
0,1964,63.23
1,1965,63.23
2,1966,59.00
3,1967,59.00
4,1968,59.00
5,1969,59.00
6,1981,450.00
7,1982,345.20
8,1983,366.16
9,1984,380.13


In [76]:
fig = px.line(
    avg_price_by_year,
    x='Year',
    y='Average_Price',
    title='Average Space Mission Launch Price Over Time',
    labels={
        'Year': 'Year',
        'Average_Price': 'Average launch price (USD millions)'
    },
    markers=True
)

fig.update_layout(
    height=550
)

HTML(fig.to_html(include_plotlyjs='cdn'))

# Chart the Number of Launches over Time by the Top 10 Organisations. 

How has the dominance of launches changed over time between the different players? 

In [77]:
launches_over_time = df_data.drop_duplicates().copy()
launches_over_time['Date'] = pd.to_datetime(
    launches_over_time['Date'],
    errors='coerce',
    utc=True
)
launches_over_time = launches_over_time.dropna(subset=['Date'])
launches_over_time['Year'] = launches_over_time['Date'].dt.year

top_10_orgs = (
    launches_over_time['Organisation']
    .value_counts()
    .head(10)
    .index
)

top_10_launches_by_year = (
    launches_over_time[launches_over_time['Organisation'].isin(top_10_orgs)]
    .groupby(['Year', 'Organisation'])
    .size()
    .reset_index(name='Launches')
)

top_10_launches_by_year.head()

,Year,Organisation,Launches
0,1957,RVSN USSR,2
1,1958,NASA,2
2,1958,RVSN USSR,5
3,1958,US Air Force,2
4,1959,General Dynamics,1


In [78]:
fig = px.line(
    top_10_launches_by_year,
    x='Year',
    y='Launches',
    color='Organisation',
    title='Number of Space Mission Launches per Year by Top 10 Organisations',
    labels={
        'Year': 'Year',
        'Launches': 'Number of launches',
        'Organisation': 'Organisation'
    },
    markers=True
)

fig.update_layout(
    height=600
)

HTML(fig.to_html(include_plotlyjs='cdn'))

# Cold War Space Race: USA vs USSR

The cold war lasted from the start of the dataset up until 1991. 

In [79]:
# Rebuild the country mapping on the FULL dataset (not df_data_clean, which
# drops ~78% of rows due to missing Price values and would badly undercount
# USSR launches, since Soviet-era missions rarely report a price).
df_map_full = df_data.drop_duplicates().copy()

country_replacements = {
    'Russia': 'Russian Federation',
    'New Mexico': 'USA',
    'Yellow Sea': 'China',
    'Shahrud Missile Test Site': 'Iran',
    'Pacific Missile Range Facility': 'USA',
    'Barents Sea': 'Russian Federation',
    'Gran Canaria': 'USA',
    'South Korea': 'Korea, Republic of',
    'North Korea': "Korea, Democratic People's Republic of",
    'USA': 'United States',
    'UK': 'United Kingdom'
}

df_map_full['Country'] = (
    df_map_full['Location']
    .str.split(',')
    .str[-1]
    .str.strip()
    .replace(country_replacements)
)

df_map_full['Date'] = pd.to_datetime(df_map_full['Date'], errors='coerce', utc=True)
df_map_full['Year'] = df_map_full['Date'].dt.year

cold_war_df = df_map_full[df_map_full['Year'] <= 1991].copy()
cold_war_df.shape

(2520, 11)

In [80]:
# Kazakhstan (home of the Baikonur Cosmodrome) launched Soviet missions,
# so it counts as USSR for this period, per the hint in the next section.
ussr_countries = ['Russian Federation', 'Kazakhstan']
usa_countries = ['United States', 'USA']

def classify_superpower(country):
    if country in ussr_countries:
        return 'USSR'
    elif country in usa_countries:
        return 'USA'
    else:
        return 'Other'

cold_war_df['Superpower'] = cold_war_df['Country'].apply(classify_superpower)
cold_war_df['Superpower'].value_counts()

Superpower
USSR     1709
USA       644
Other     167
Name: count, dtype: int64

## Create a Plotly Pie Chart comparing the total number of launches of the USSR and the USA

Hint: Remember to include former Soviet Republics like Kazakhstan when analysing the total number of launches. 

In [81]:
superpower_launches = (
    cold_war_df[cold_war_df['Superpower'].isin(['USA', 'USSR'])]
    .groupby('Superpower')
    .size()
    .reset_index(name='Launches')
)

superpower_launches

,Superpower,Launches
0,USA,644
1,USSR,1709


In [82]:
fig = px.pie(
    superpower_launches,
    names='Superpower',
    values='Launches',
    title='Cold War Space Race: Total Launches, USA vs USSR (up to 1991)',
    color='Superpower',
    color_discrete_map={'USA': '#3d5a80', 'USSR': '#b23a48'},
    hole=0.35
)

fig.update_traces(textinfo='percent+label+value')
fig.update_layout(height=550)

HTML(fig.to_html(include_plotlyjs='cdn'))

## Create a Chart that Shows the Total Number of Launches Year-On-Year by the Two Superpowers

In [83]:
yearly_superpower_launches = (
    cold_war_df[cold_war_df['Superpower'].isin(['USA', 'USSR'])]
    .groupby(['Year', 'Superpower'])
    .size()
    .reset_index(name='Launches')
)
yearly_superpower_launches['Year'] = yearly_superpower_launches['Year'].astype(int)

yearly_superpower_launches.head()

,Year,Superpower,Launches
0,1957,USA,1
1,1957,USSR,2
2,1958,USA,17
3,1958,USSR,5
4,1959,USA,16


In [84]:
fig = px.line(
    yearly_superpower_launches,
    x='Year',
    y='Launches',
    color='Superpower',
    title='Cold War Space Race: Launches per Year, USA vs USSR',
    labels={
        'Year': 'Year',
        'Launches': 'Number of launches',
        'Superpower': 'Superpower'
    },
    color_discrete_map={'USA': '#3d5a80', 'USSR': '#b23a48'},
    markers=True
)

fig.update_layout(
    height=550
)

HTML(fig.to_html(include_plotlyjs='cdn'))

## Chart the Total Number of Mission Failures Year on Year.

In [85]:
# Mission_Status has 4 values: 'Success', 'Failure', 'Partial Failure',
# 'Prelaunch Failure' — treat anything containing 'Failure' as a failure
cold_war_failures = cold_war_df[cold_war_df['Superpower'].isin(['USA', 'USSR'])].copy()
cold_war_failures['Is_Failure'] = cold_war_failures['Mission_Status'].str.contains('Failure', na=False)

yearly_failures = (
    cold_war_failures[cold_war_failures['Is_Failure']]
    .groupby(['Year', 'Superpower'])
    .size()
    .reset_index(name='Failures')
)
yearly_failures['Year'] = yearly_failures['Year'].astype(int)

yearly_failures.head()

,Year,Superpower,Failures
0,1957,USA,1
1,1958,USA,12
2,1958,USSR,4
3,1959,USA,10
4,1959,USSR,2


In [86]:
fig = px.bar(
    yearly_failures,
    x='Year',
    y='Failures',
    color='Superpower',
    barmode='group',
    title='Cold War Space Race: Mission Failures per Year, USA vs USSR',
    labels={
        'Year': 'Year',
        'Failures': 'Number of failed missions',
        'Superpower': 'Superpower'
    },
    color_discrete_map={'USA': '#3d5a80', 'USSR': '#b23a48'}
)

fig.update_layout(
    height=550,
    bargap=0.15
)

HTML(fig.to_html(include_plotlyjs='cdn'))

## Chart the Percentage of Failures over Time

Did failures go up or down over time? Did the countries get better at minimising risk and improving their chances of success over time? 

In [87]:
cold_war_superpowers = cold_war_df[cold_war_df['Superpower'].isin(['USA', 'USSR'])].copy()
cold_war_superpowers['Is_Failure'] = cold_war_superpowers['Mission_Status'].str.contains('Failure', na=False)

yearly_totals = (
    cold_war_superpowers
    .groupby(['Year', 'Superpower'])
    .agg(
        Total_Launches=('Mission_Status', 'size'),
        Failures=('Is_Failure', 'sum')
    )
    .reset_index()
)
yearly_totals['Year'] = yearly_totals['Year'].astype(int)
yearly_totals['Failure_Rate'] = (
    yearly_totals['Failures'] / yearly_totals['Total_Launches']
) * 100

yearly_totals.head()

,Year,Superpower,Total_Launches,Failures,Failure_Rate
0,1957,USA,1,1,100.00
1,1957,USSR,2,0,0.00
2,1958,USA,17,12,70.59
3,1958,USSR,5,4,80.00
4,1959,USA,16,10,62.50


In [88]:
overall_rates = (
    cold_war_superpowers
    .groupby('Superpower')
    .agg(Total=('Is_Failure', 'size'), Failures=('Is_Failure', 'sum'))
)
overall_rates['Failure_Rate'] = overall_rates['Failures'] / overall_rates['Total'] * 100
overall_rates

,Total,Failures,Failure_Rate
Superpower,,,
USA,644,114,17.70
USSR,1709,152,8.89


In [89]:
fig = px.line(
    yearly_totals,
    x='Year',
    y='Failure_Rate',
    color='Superpower',
    title='Cold War Space Race: Mission Failure Rate per Year, USA vs USSR',
    labels={
        'Year': 'Year',
        'Failure_Rate': 'Failure rate (%)',
        'Superpower': 'Superpower'
    },
    color_discrete_map={'USA': '#3d5a80', 'USSR': '#b23a48'},
    markers=True
)

fig.update_layout(
    height=550,
    yaxis_range=[0, 100]
)

HTML(fig.to_html(include_plotlyjs='cdn'))

# For Every Year Show which Country was in the Lead in terms of Total Number of Launches up to and including including 2020)

Do the results change if we only look at the number of successful launches? 

In [90]:
launches_up_to_2020 = df_map_full[df_map_full['Year'] <= 2020].copy()

country_year_counts = (
    launches_up_to_2020
    .groupby(['Year', 'Country'])
    .size()
    .reset_index(name='Launches')
)

cumulative_launches = (
    country_year_counts
    .pivot(index='Year', columns='Country', values='Launches')
    .fillna(0)
    .cumsum()
)

leading_country_total = (
    cumulative_launches
    .idxmax(axis=1)
    .reset_index(name='Leading_Country')
)

leading_country_total.tail(10)

,Year,Leading_Country
54,"2,011.00",Russian Federation
55,"2,012.00",Russian Federation
56,"2,013.00",Russian Federation
57,"2,014.00",Russian Federation
58,"2,015.00",Russian Federation
59,"2,016.00",Russian Federation
60,"2,017.00",Russian Federation
61,"2,018.00",Russian Federation
62,"2,019.00",Russian Federation
63,"2,020.00",Russian Federation


In [91]:
launches_up_to_2020['Is_Success'] = launches_up_to_2020['Mission_Status'] == 'Success'
success_only = launches_up_to_2020[launches_up_to_2020['Is_Success']]

country_year_success = (
    success_only
    .groupby(['Year', 'Country'])
    .size()
    .reset_index(name='Successful_Launches')
)

cumulative_success = (
    country_year_success
    .pivot(index='Year', columns='Country', values='Successful_Launches')
    .fillna(0)
    .cumsum()
)

leading_country_success = (
    cumulative_success
    .idxmax(axis=1)
    .reset_index(name='Leading_Country_Success')
)

# Compare the two rankings
leader_comparison = leading_country_total.merge(leading_country_success, on='Year')
years_where_leader_differs = leader_comparison[
    leader_comparison['Leading_Country'] != leader_comparison['Leading_Country_Success']
]

years_where_leader_differs

,Year,Leading_Country,Leading_Country_Success
15,"1,972.00",United States,Russian Federation
16,"1,973.00",United States,Russian Federation


# Create a Year-on-Year Chart Showing the Organisation Doing the Most Number of Launches

Which organisation was dominant in the 1970s and 1980s? Which organisation was dominant in 2018, 2019 and 2020? 

In [93]:
launches_full = df_data.drop_duplicates().copy()
launches_full['Date'] = pd.to_datetime(launches_full['Date'], errors='coerce', utc=True)
launches_full = launches_full.dropna(subset=['Date'])
launches_full['Year'] = launches_full['Date'].dt.year

In [94]:
org_year_counts = (
    launches_full
    .groupby(['Year', 'Organisation'])
    .size()
    .reset_index(name='Launches')
)

dominant_org_by_year = (
    org_year_counts
    .sort_values('Launches', ascending=False)
    .drop_duplicates(subset='Year')
    .sort_values('Year')
    .reset_index(drop=True)
)

dominant_org_by_year

,Year,Organisation,Launches
0,1957,RVSN USSR,2
1,1958,AMBA,7
2,1959,US Air Force,10
3,1960,US Air Force,21
4,1961,US Air Force,23
...,...,...,...
59,2016,CASC,22
60,2017,SpaceX,18
61,2018,CASC,37
62,2019,CASC,27


In [95]:
fig = px.bar(
    dominant_org_by_year,
    x='Year',
    y='Launches',
    color='Organisation',
    title='Dominant Organisation by Launches per Year',
    labels={
        'Year': 'Year',
        'Launches': 'Number of launches (leading organisation)',
        'Organisation': 'Organisation'
    }
)

fig.update_layout(
    height=550,
    bargap=0.1
)

HTML(fig.to_html(include_plotlyjs='cdn'))